# Home Credit Temporal and Deterioration Analysis

This notebook analyses the temporal structure of historical Home Credit accounts to identify recent delinquency, historical delinquency, repayment deterioration, utilization behaviour and activity recency.

The analysis uses historical monthly records associated with the current application and retains the standard application-level `TARGET` only for portfolio-level risk interpretation.

No future monthly default target is manufactured.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

HOME_CREDIT_RAW_PATH = (
    "/Volumes/workspace/default/home_credit_raw"
)

BRONZE_PATH = (
    f"{HOME_CREDIT_RAW_PATH}/bronze"
)

GOLD_PATH = (
    f"{HOME_CREDIT_RAW_PATH}/gold_home_credit_application"
)

TEMPORAL_OUTPUT_PATH = (
    f"{HOME_CREDIT_RAW_PATH}/"
    "home_credit_temporal_deterioration"
)

# ------------------------------------------------------------
# Load application-level Gold dataset
# ------------------------------------------------------------

hc_gold = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

# ------------------------------------------------------------
# Load historical monthly source tables
# ------------------------------------------------------------

bureau_balance = (
    spark.read
    .format("delta")
    .load(
        f"{BRONZE_PATH}/bureau_balance"
    )
)

bureau = (
    spark.read
    .format("delta")
    .load(
        f"{BRONZE_PATH}/bureau"
    )
)

installments_payments = (
    spark.read
    .format("delta")
    .load(
        f"{BRONZE_PATH}/installments_payments"
    )
)

pos_cash_balance = (
    spark.read
    .format("delta")
    .load(
        f"{BRONZE_PATH}/POS_CASH_balance"
    )
)

credit_card_balance = (
    spark.read
    .format("delta")
    .load(
        f"{BRONZE_PATH}/credit_card_balance"
    )
)

# ------------------------------------------------------------
# Validate required keys
# ------------------------------------------------------------

required_columns = {
    "bureau_balance": [
        "SK_ID_BUREAU",
        "MONTHS_BALANCE",
        "STATUS"
    ],
    "bureau": [
        "SK_ID_BUREAU",
        "SK_ID_CURR"
    ],
    "installments_payments": [
        "SK_ID_PREV",
        "SK_ID_CURR",
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT",
        "AMT_INSTALMENT",
        "AMT_PAYMENT"
    ],
    "pos_cash_balance": [
        "SK_ID_PREV",
        "SK_ID_CURR",
        "MONTHS_BALANCE",
        "SK_DPD",
        "SK_DPD_DEF"
    ],
    "credit_card_balance": [
        "SK_ID_PREV",
        "SK_ID_CURR",
        "MONTHS_BALANCE",
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "SK_DPD",
        "SK_DPD_DEF"
    ]
}

loaded_tables = {
    "bureau_balance": bureau_balance,
    "bureau": bureau,
    "installments_payments": installments_payments,
    "pos_cash_balance": pos_cash_balance,
    "credit_card_balance": credit_card_balance
}

for table_name, required in required_columns.items():

    missing = [
        column_name
        for column_name in required
        if column_name not in loaded_tables[table_name].columns
    ]

    if missing:
        raise ValueError(
            f"{table_name} is missing required columns: {missing}"
        )

# ------------------------------------------------------------
# Basic source-size audit
# ------------------------------------------------------------

source_counts = [
    (
        "application_gold",
        hc_gold.count()
    ),
    (
        "bureau",
        bureau.count()
    ),
    (
        "bureau_balance",
        bureau_balance.count()
    ),
    (
        "installments_payments",
        installments_payments.count()
    ),
    (
        "pos_cash_balance",
        pos_cash_balance.count()
    ),
    (
        "credit_card_balance",
        credit_card_balance.count()
    )
]

source_count_df = spark.createDataFrame(
    source_counts,
    ["source_table", "row_count"]
)

print("=" * 70)
print("HOME CREDIT TEMPORAL DATASET AUDIT")
print("=" * 70)

source_count_df.show(truncate=False)

print("\nGold applications:", f"{hc_gold.count():,}")

print("\nTemporal source tables loaded successfully.")

HOME CREDIT TEMPORAL DATASET AUDIT
+---------------------+---------+
|source_table         |row_count|
+---------------------+---------+
|application_gold     |307511   |
|bureau               |1716428  |
|bureau_balance       |27299925 |
|installments_payments|13605401 |
|pos_cash_balance     |10001358 |
|credit_card_balance  |3840312  |
+---------------------+---------+


Gold applications: 307,511

Temporal source tables loaded successfully.


In [0]:
# Build temporal histories and establish observation windows

# -------------------------------------------------------------------
# Bureau monthly history
# -------------------------------------------------------------------

bureau_history = (
    bureau_balance
    .join(
        bureau.select("SK_ID_BUREAU", "SK_ID_CURR"),
        on="SK_ID_BUREAU",
        how="inner"
    )
    .select(
        "SK_ID_CURR",
        "SK_ID_BUREAU",
        "MONTHS_BALANCE",
        "STATUS"
    )
    .withColumn(
        "status_clean",
        F.upper(F.trim(F.col("STATUS")))
    )
    .withColumn(
        "bureau_month_index",
        -F.col("MONTHS_BALANCE")
    )
)

bureau_history_summary = (
    bureau_history
    .groupBy("SK_ID_CURR")
    .agg(
        F.countDistinct("SK_ID_BUREAU").alias("bureau_account_count"),
        F.count("*").alias("bureau_month_record_count"),
        F.countDistinct("MONTHS_BALANCE").alias("bureau_month_count"),
        F.min("MONTHS_BALANCE").alias("bureau_oldest_month"),
        F.max("MONTHS_BALANCE").alias("bureau_most_recent_month"),
        F.max(
            F.when(
                F.col("status_clean").isin("1", "2", "3", "4", "5"),
                F.col("bureau_month_index")
            )
        ).alias("bureau_most_recent_delinquency_month"),
        F.max(
            F.when(
                F.col("status_clean").isin("2", "3", "4", "5"),
                F.col("bureau_month_index")
            )
        ).alias("bureau_most_recent_severe_delinquency_month"),
        F.sum(
            F.when(
                F.col("status_clean").isin("1", "2", "3", "4", "5"),
                1
            ).otherwise(0)
        ).alias("bureau_delinquent_month_records"),
        F.sum(
            F.when(
                F.col("status_clean").isin("2", "3", "4", "5"),
                1
            ).otherwise(0)
        ).alias("bureau_severe_delinquent_month_records"),
        F.sum(
            F.when(
                F.col("status_clean") == "5",
                1
            ).otherwise(0)
        ).alias("bureau_status_5_month_records")
    )
)

# -------------------------------------------------------------------
# Installment payment history
# -------------------------------------------------------------------

installment_history = (
    installments_payments
    .select(
        "SK_ID_CURR",
        "SK_ID_PREV",
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT",
        "AMT_INSTALMENT",
        "AMT_PAYMENT"
    )
    .withColumn(
        "payment_delay_days",
        F.greatest(
            F.col("DAYS_ENTRY_PAYMENT") - F.col("DAYS_INSTALMENT"),
            F.lit(0)
        )
    )
    .withColumn(
        "installment_shortfall",
        F.greatest(
            F.col("AMT_INSTALMENT") - F.col("AMT_PAYMENT"),
            F.lit(0.0)
        )
    )
    .withColumn(
        "payment_completed_flag",
        F.when(
            F.col("AMT_PAYMENT") >= F.col("AMT_INSTALMENT"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "late_payment_flag",
        F.when(F.col("payment_delay_days") > 0, 1).otherwise(0)
    )
)

installment_temporal_summary = (
    installment_history
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("installment_payment_records"),
        F.countDistinct("SK_ID_PREV").alias("installment_account_count"),
        F.min("DAYS_INSTALMENT").alias("installment_oldest_day"),
        F.max("DAYS_INSTALMENT").alias("installment_most_recent_day"),
        F.avg("payment_delay_days").alias("historical_avg_payment_delay"),
        F.max("payment_delay_days").alias("historical_max_payment_delay"),
        F.sum("late_payment_flag").alias("historical_late_payment_records"),
        F.sum("payment_completed_flag").alias("historical_completed_payment_records"),
        F.sum("installment_shortfall").alias("historical_payment_shortfall")
    )
)

# -------------------------------------------------------------------
# POS-CASH monthly history
# -------------------------------------------------------------------

pos_temporal = (
    pos_cash_balance
    .select(
        "SK_ID_CURR",
        "SK_ID_PREV",
        "MONTHS_BALANCE",
        "SK_DPD",
        "SK_DPD_DEF"
    )
    .withColumn(
        "month_index",
        -F.col("MONTHS_BALANCE")
    )
    .withColumn(
        "dpd_flag",
        F.when(F.col("SK_DPD") > 0, 1).otherwise(0)
    )
    .withColumn(
        "severe_dpd_flag",
        F.when(F.col("SK_DPD_DEF") > 0, 1).otherwise(0)
    )
)

pos_temporal_summary = (
    pos_temporal
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("pos_month_records"),
        F.countDistinct("SK_ID_PREV").alias("pos_account_count"),
        F.countDistinct("MONTHS_BALANCE").alias("pos_distinct_months"),
        F.min("MONTHS_BALANCE").alias("pos_oldest_month"),
        F.max("MONTHS_BALANCE").alias("pos_most_recent_month"),
        F.sum("dpd_flag").alias("pos_dpd_month_records"),
        F.sum("severe_dpd_flag").alias("pos_severe_dpd_month_records"),
        F.max("SK_DPD").alias("pos_max_dpd"),
        F.max("SK_DPD_DEF").alias("pos_max_severe_dpd")
    )
)

# -------------------------------------------------------------------
# Credit-card monthly history
# -------------------------------------------------------------------

cc_temporal = (
    credit_card_balance
    .select(
        "SK_ID_CURR",
        "SK_ID_PREV",
        "MONTHS_BALANCE",
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "SK_DPD",
        "SK_DPD_DEF"
    )
    .withColumn(
        "month_index",
        -F.col("MONTHS_BALANCE")
    )
    .withColumn(
        "dpd_flag",
        F.when(F.col("SK_DPD") > 0, 1).otherwise(0)
    )
    .withColumn(
        "severe_dpd_flag",
        F.when(F.col("SK_DPD_DEF") > 0, 1).otherwise(0)
    )
    .withColumn(
        "utilization_ratio",
        F.when(
            F.col("AMT_CREDIT_LIMIT_ACTUAL") > 0,
            F.least(
                F.col("AMT_BALANCE") / F.col("AMT_CREDIT_LIMIT_ACTUAL"),
                F.lit(1.0)
            )
        )
    )
)

cc_temporal_summary = (
    cc_temporal
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("cc_month_records"),
        F.countDistinct("SK_ID_PREV").alias("cc_account_count"),
        F.countDistinct("MONTHS_BALANCE").alias("cc_distinct_months"),
        F.min("MONTHS_BALANCE").alias("cc_oldest_month"),
        F.max("MONTHS_BALANCE").alias("cc_most_recent_month"),
        F.sum("dpd_flag").alias("cc_dpd_month_records"),
        F.sum("severe_dpd_flag").alias("cc_severe_dpd_month_records"),
        F.max("SK_DPD").alias("cc_max_dpd"),
        F.max("SK_DPD_DEF").alias("cc_max_severe_dpd"),
        F.avg("utilization_ratio").alias("historical_avg_cc_utilization"),
        F.max("utilization_ratio").alias("historical_max_cc_utilization")
    )
)

# -------------------------------------------------------------------
# Combine customer-level temporal summaries with application Gold
# -------------------------------------------------------------------

temporal_summary = (
    hc_gold
    .select("SK_ID_CURR", "TARGET")
    .join(bureau_history_summary, "SK_ID_CURR", "left")
    .join(installment_temporal_summary, "SK_ID_CURR", "left")
    .join(pos_temporal_summary, "SK_ID_CURR", "left")
    .join(cc_temporal_summary, "SK_ID_CURR", "left")
)

# -------------------------------------------------------------------
# Derived recency and deterioration indicators
# -------------------------------------------------------------------

temporal_summary = (
    temporal_summary
    .withColumn(
        "bureau_delinquency_recency_months",
        F.col("bureau_most_recent_delinquency_month")
    )
    .withColumn(
        "bureau_severe_delinquency_recency_months",
        F.col("bureau_most_recent_severe_delinquency_month")
    )
    .withColumn(
        "bureau_delinquency_rate",
        F.when(
            F.col("bureau_month_record_count") > 0,
            F.col("bureau_delinquent_month_records") /
            F.col("bureau_month_record_count")
        )
    )
    .withColumn(
        "bureau_severe_delinquency_rate",
        F.when(
            F.col("bureau_month_record_count") > 0,
            F.col("bureau_severe_delinquent_month_records") /
            F.col("bureau_month_record_count")
        )
    )
    .withColumn(
        "pos_dpd_rate",
        F.when(
            F.col("pos_month_records") > 0,
            F.col("pos_dpd_month_records") /
            F.col("pos_month_records")
        )
    )
    .withColumn(
        "pos_severe_dpd_rate",
        F.when(
            F.col("pos_month_records") > 0,
            F.col("pos_severe_dpd_month_records") /
            F.col("pos_month_records")
        )
    )
    .withColumn(
        "cc_dpd_rate",
        F.when(
            F.col("cc_month_records") > 0,
            F.col("cc_dpd_month_records") /
            F.col("cc_month_records")
        )
    )
    .withColumn(
        "cc_severe_dpd_rate",
        F.when(
            F.col("cc_month_records") > 0,
            F.col("cc_severe_dpd_month_records") /
            F.col("cc_month_records")
        )
    )
    .withColumn(
        "installment_late_payment_rate",
        F.when(
            F.col("installment_payment_records") > 0,
            F.col("historical_late_payment_records") /
            F.col("installment_payment_records")
        )
    )
    .withColumn(
        "installment_completion_rate",
        F.when(
            F.col("installment_payment_records") > 0,
            F.col("historical_completed_payment_records") /
            F.col("installment_payment_records")
        )
    )
    .withColumn(
        "temporal_history_available",
        F.when(
            (
                F.col("bureau_month_record_count").isNotNull()
                | F.col("installment_payment_records").isNotNull()
                | F.col("pos_month_records").isNotNull()
                | F.col("cc_month_records").isNotNull()
            ),
            1
        ).otherwise(0)
    )
)

# -------------------------------------------------------------------
# Coverage and temporal-window diagnostics
# -------------------------------------------------------------------

coverage_metrics = (
    temporal_summary
    .agg(
        F.count("*").alias("applications"),
        F.sum(
            F.when(F.col("bureau_month_record_count").isNotNull(), 1).otherwise(0)
        ).alias("with_bureau_history"),
        F.sum(
            F.when(F.col("installment_payment_records").isNotNull(), 1).otherwise(0)
        ).alias("with_installment_history"),
        F.sum(
            F.when(F.col("pos_month_records").isNotNull(), 1).otherwise(0)
        ).alias("with_pos_history"),
        F.sum(
            F.when(F.col("cc_month_records").isNotNull(), 1).otherwise(0)
        ).alias("with_cc_history"),
        F.sum("temporal_history_available").alias("with_any_temporal_history")
    )
)

print("=" * 70)
print("CUSTOMER-LEVEL TEMPORAL HISTORY SUMMARY")
print("=" * 70)

print("\nCoverage:")
coverage_metrics.show(truncate=False)

print("\nTemporal feature summary:")
(
    temporal_summary
    .select(
        "SK_ID_CURR",
        "TARGET",
        "bureau_account_count",
        "bureau_month_record_count",
        "bureau_delinquency_rate",
        "bureau_severe_delinquency_rate",
        "installment_payment_records",
        "historical_avg_payment_delay",
        "installment_late_payment_rate",
        "installment_completion_rate",
        "pos_month_records",
        "pos_dpd_rate",
        "pos_severe_dpd_rate",
        "cc_month_records",
        "cc_dpd_rate",
        "cc_severe_dpd_rate",
        "historical_avg_cc_utilization",
        "historical_max_cc_utilization"
    )
    .describe()
    .show(truncate=False)
)

print("\nMost recent historical activity distributions:")
(
    temporal_summary
    .select(
        "bureau_most_recent_month",
        "pos_most_recent_month",
        "cc_most_recent_month",
        "installment_most_recent_day"
    )
    .describe()
    .show(truncate=False)
)

CUSTOMER-LEVEL TEMPORAL HISTORY SUMMARY

Coverage:
+------------+-------------------+------------------------+----------------+---------------+-------------------------+
|applications|with_bureau_history|with_installment_history|with_pos_history|with_cc_history|with_any_temporal_history|
+------------+-------------------+------------------------+----------------+---------------+-------------------------+
|307511      |92231              |291643                  |289444          |86905          |295494                   |
+------------+-------------------+------------------------+----------------+---------------+-------------------------+


Temporal feature summary:
+-------+------------------+-------------------+--------------------+-------------------------+-----------------------+------------------------------+---------------------------+----------------------------+-----------------------------+---------------------------+------------------+--------------------+--------------------+

In [0]:
# -------------------------------------------------------------------
# Bureau deterioration
# -------------------------------------------------------------------

bureau_recent_corrected = (
    bureau_history
    .withColumn(
        "recent_3m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -2, 1).otherwise(0)
    )
    .withColumn(
        "recent_6m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -5, 1).otherwise(0)
    )
    .withColumn(
        "delinquency_flag",
        F.when(
            F.col("status_clean").isin("1", "2", "3", "4", "5"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "severe_delinquency_flag",
        F.when(
            F.col("status_clean").isin("2", "3", "4", "5"),
            1
        ).otherwise(0)
    )
)

bureau_deterioration_corrected = (
    bureau_recent_corrected
    .groupBy("SK_ID_CURR")
    .agg(
        F.avg("delinquency_flag").alias(
            "bureau_historical_delinquency_rate"
        ),
        F.avg("severe_delinquency_flag").alias(
            "bureau_historical_severe_delinquency_rate"
        ),

        F.sum(
            F.when(F.col("recent_3m_flag") == 1, 1).otherwise(0)
        ).alias("bureau_recent_3m_records"),

        F.sum(
            F.when(F.col("recent_6m_flag") == 1, 1).otherwise(0)
        ).alias("bureau_recent_6m_records"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("delinquency_flag") == 1),
                1
            ).otherwise(0)
        ).alias("bureau_recent_3m_delinquent_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("delinquency_flag") == 1),
                1
            ).otherwise(0)
        ).alias("bureau_recent_6m_delinquent_records"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("severe_delinquency_flag") == 1),
                1
            ).otherwise(0)
        ).alias("bureau_recent_3m_severe_delinquent_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("severe_delinquency_flag") == 1),
                1
            ).otherwise(0)
        ).alias("bureau_recent_6m_severe_delinquent_records")
    )
    .withColumn(
        "bureau_recent_3m_delinquency_rate",
        F.when(
            F.col("bureau_recent_3m_records") > 0,
            F.col("bureau_recent_3m_delinquent_records") /
            F.col("bureau_recent_3m_records")
        )
    )
    .withColumn(
        "bureau_recent_6m_delinquency_rate",
        F.when(
            F.col("bureau_recent_6m_records") > 0,
            F.col("bureau_recent_6m_delinquent_records") /
            F.col("bureau_recent_6m_records")
        )
    )
    .withColumn(
        "bureau_recent_3m_severe_delinquency_rate",
        F.when(
            F.col("bureau_recent_3m_records") > 0,
            F.col("bureau_recent_3m_severe_delinquent_records") /
            F.col("bureau_recent_3m_records")
        )
    )
    .withColumn(
        "bureau_recent_6m_severe_delinquency_rate",
        F.when(
            F.col("bureau_recent_6m_records") > 0,
            F.col("bureau_recent_6m_severe_delinquent_records") /
            F.col("bureau_recent_6m_records")
        )
    )
    .withColumn(
        "bureau_delinquency_deterioration",
        F.col("bureau_recent_3m_delinquency_rate")
        - F.col("bureau_historical_delinquency_rate")
    )
    .withColumn(
        "bureau_severe_delinquency_deterioration",
        F.col("bureau_recent_3m_severe_delinquency_rate")
        - F.col("bureau_historical_severe_delinquency_rate")
    )
)

# -------------------------------------------------------------------
# POS-CASH deterioration
# -------------------------------------------------------------------

pos_deterioration_corrected = (
    pos_temporal
    .withColumn(
        "recent_3m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -2, 1).otherwise(0)
    )
    .withColumn(
        "recent_6m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -5, 1).otherwise(0)
    )
    .groupBy("SK_ID_CURR")
    .agg(
        F.avg("dpd_flag").alias("pos_historical_dpd_rate"),
        F.avg("severe_dpd_flag").alias("pos_historical_severe_dpd_rate"),

        F.sum(
            F.when(F.col("recent_3m_flag") == 1, 1).otherwise(0)
        ).alias("pos_recent_3m_records"),

        F.sum(
            F.when(F.col("recent_6m_flag") == 1, 1).otherwise(0)
        ).alias("pos_recent_6m_records"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("pos_recent_3m_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("pos_recent_6m_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("severe_dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("pos_recent_3m_severe_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("severe_dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("pos_recent_6m_severe_dpd_records")
    )
    .withColumn(
        "pos_recent_3m_dpd_rate",
        F.when(
            F.col("pos_recent_3m_records") > 0,
            F.col("pos_recent_3m_dpd_records") /
            F.col("pos_recent_3m_records")
        )
    )
    .withColumn(
        "pos_recent_6m_dpd_rate",
        F.when(
            F.col("pos_recent_6m_records") > 0,
            F.col("pos_recent_6m_dpd_records") /
            F.col("pos_recent_6m_records")
        )
    )
    .withColumn(
        "pos_recent_3m_severe_dpd_rate",
        F.when(
            F.col("pos_recent_3m_records") > 0,
            F.col("pos_recent_3m_severe_dpd_records") /
            F.col("pos_recent_3m_records")
        )
    )
    .withColumn(
        "pos_recent_6m_severe_dpd_rate",
        F.when(
            F.col("pos_recent_6m_records") > 0,
            F.col("pos_recent_6m_severe_dpd_records") /
            F.col("pos_recent_6m_records")
        )
    )
    .withColumn(
        "pos_dpd_deterioration",
        F.col("pos_recent_3m_dpd_rate")
        - F.col("pos_historical_dpd_rate")
    )
    .withColumn(
        "pos_severe_dpd_deterioration",
        F.col("pos_recent_3m_severe_dpd_rate")
        - F.col("pos_historical_severe_dpd_rate")
    )
)

# -------------------------------------------------------------------
# Credit-card deterioration
# -------------------------------------------------------------------

cc_deterioration_corrected = (
    cc_temporal
    .withColumn(
        "recent_3m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -2, 1).otherwise(0)
    )
    .withColumn(
        "recent_6m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -5, 1).otherwise(0)
    )
    .withColumn(
        "recent_12m_flag",
        F.when(F.col("MONTHS_BALANCE") >= -11, 1).otherwise(0)
    )
    .groupBy("SK_ID_CURR")
    .agg(
        F.avg("utilization_ratio").alias(
            "cc_historical_avg_utilization"
        ),
        F.avg("dpd_flag").alias(
            "cc_historical_dpd_rate"
        ),
        F.avg("severe_dpd_flag").alias(
            "cc_historical_severe_dpd_rate"
        ),

        F.sum(
            F.when(F.col("recent_3m_flag") == 1, 1).otherwise(0)
        ).alias("cc_recent_3m_records"),

        F.sum(
            F.when(F.col("recent_6m_flag") == 1, 1).otherwise(0)
        ).alias("cc_recent_6m_records"),

        F.avg(
            F.when(
                F.col("recent_3m_flag") == 1,
                F.col("utilization_ratio")
            )
        ).alias("cc_recent_3m_avg_utilization"),

        F.avg(
            F.when(
                F.col("recent_6m_flag") == 1,
                F.col("utilization_ratio")
            )
        ).alias("cc_recent_6m_avg_utilization"),

        F.avg(
            F.when(
                F.col("recent_12m_flag") == 1,
                F.col("utilization_ratio")
            )
        ).alias("cc_recent_12m_avg_utilization"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("cc_recent_3m_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("cc_recent_6m_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_3m_flag") == 1) &
                (F.col("severe_dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("cc_recent_3m_severe_dpd_records"),

        F.sum(
            F.when(
                (F.col("recent_6m_flag") == 1) &
                (F.col("severe_dpd_flag") == 1),
                1
            ).otherwise(0)
        ).alias("cc_recent_6m_severe_dpd_records")
    )
    .withColumn(
        "cc_recent_3m_dpd_rate",
        F.when(
            F.col("cc_recent_3m_records") > 0,
            F.col("cc_recent_3m_dpd_records") /
            F.col("cc_recent_3m_records")
        )
    )
    .withColumn(
        "cc_recent_6m_dpd_rate",
        F.when(
            F.col("cc_recent_6m_records") > 0,
            F.col("cc_recent_6m_dpd_records") /
            F.col("cc_recent_6m_records")
        )
    )
    .withColumn(
        "cc_recent_3m_severe_dpd_rate",
        F.when(
            F.col("cc_recent_3m_records") > 0,
            F.col("cc_recent_3m_severe_dpd_records") /
            F.col("cc_recent_3m_records")
        )
    )
    .withColumn(
        "cc_recent_6m_severe_dpd_rate",
        F.when(
            F.col("cc_recent_6m_records") > 0,
            F.col("cc_recent_6m_severe_dpd_records") /
            F.col("cc_recent_6m_records")
        )
    )
    .withColumn(
        "cc_utilization_deterioration",
        F.col("cc_recent_3m_avg_utilization")
        - F.col("cc_historical_avg_utilization")
    )
    .withColumn(
        "cc_dpd_deterioration",
        F.col("cc_recent_3m_dpd_rate")
        - F.col("cc_historical_dpd_rate")
    )
    .withColumn(
        "cc_severe_dpd_deterioration",
        F.col("cc_recent_3m_severe_dpd_rate")
        - F.col("cc_historical_severe_dpd_rate")
    )
)

# -------------------------------------------------------------------
# Rebuild the combined temporal deterioration dataset
# -------------------------------------------------------------------

temporal_deterioration = (
    temporal_summary
    .join(bureau_deterioration_corrected, "SK_ID_CURR", "left")
    .join(installment_deterioration, "SK_ID_CURR", "left")
    .join(pos_deterioration_corrected, "SK_ID_CURR", "left")
    .join(cc_deterioration_corrected, "SK_ID_CURR", "left")
)

# -------------------------------------------------------------------
# Clean credit-card utilization and construct indicators
# -------------------------------------------------------------------

temporal_deterioration = (
    temporal_deterioration
    .withColumn(
        "cc_historical_avg_utilization_clean",
        F.when(
            F.col("cc_historical_avg_utilization").isNotNull(),
            F.greatest(
                F.lit(0.0),
                F.least(
                    F.lit(1.0),
                    F.col("cc_historical_avg_utilization")
                )
            )
        )
    )
    .withColumn(
        "cc_recent_3m_avg_utilization_clean",
        F.when(
            F.col("cc_recent_3m_avg_utilization").isNotNull(),
            F.greatest(
                F.lit(0.0),
                F.least(
                    F.lit(1.0),
                    F.col("cc_recent_3m_avg_utilization")
                )
            )
        )
    )
    .withColumn(
        "cc_utilization_deterioration_clean",
        F.when(
            F.col("cc_recent_3m_avg_utilization_clean").isNotNull() &
            F.col("cc_historical_avg_utilization_clean").isNotNull(),
            F.col("cc_recent_3m_avg_utilization_clean")
            - F.col("cc_historical_avg_utilization_clean")
        )
    )
    .withColumn(
        "any_recent_delinquency_flag",
        F.when(
            (
                (F.coalesce(
                    F.col("bureau_recent_3m_delinquent_records"),
                    F.lit(0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("pos_recent_3m_dpd_records"),
                    F.lit(0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("cc_recent_3m_dpd_records"),
                    F.lit(0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("installment_recent_3m_late_count"),
                    F.lit(0)
                ) > 0)
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "recent_deterioration_flag",
        F.when(
            (
                (F.coalesce(
                    F.col("bureau_delinquency_deterioration"),
                    F.lit(0.0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("pos_dpd_deterioration"),
                    F.lit(0.0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("cc_dpd_deterioration"),
                    F.lit(0.0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("installment_late_payment_deterioration"),
                    F.lit(0.0)
                ) > 0)
                |
                (F.coalesce(
                    F.col("cc_utilization_deterioration_clean"),
                    F.lit(0.0)
                ) > 0)
            ),
            1
        ).otherwise(0)
    )
)

# -------------------------------------------------------------------
# Quality checks
# -------------------------------------------------------------------

rate_range_check = (
    temporal_deterioration
    .agg(
        F.min("bureau_recent_3m_delinquency_rate").alias(
            "min_bureau_recent_3m_rate"
        ),
        F.max("bureau_recent_3m_delinquency_rate").alias(
            "max_bureau_recent_3m_rate"
        ),
        F.min("bureau_recent_6m_delinquency_rate").alias(
            "min_bureau_recent_6m_rate"
        ),
        F.max("bureau_recent_6m_delinquency_rate").alias(
            "max_bureau_recent_6m_rate"
        ),
        F.min("pos_recent_3m_dpd_rate").alias(
            "min_pos_recent_3m_rate"
        ),
        F.max("pos_recent_3m_dpd_rate").alias(
            "max_pos_recent_3m_rate"
        ),
        F.min("cc_recent_3m_dpd_rate").alias(
            "min_cc_recent_3m_rate"
        ),
        F.max("cc_recent_3m_dpd_rate").alias(
            "max_cc_recent_3m_rate"
        ),
        F.min("installment_recent_3m_late_rate").alias(
            "min_installment_recent_3m_rate"
        ),
        F.max("installment_recent_3m_late_rate").alias(
            "max_installment_recent_3m_rate"
        )
    )
)

invalid_rates = (
    temporal_deterioration
    .filter(
        (F.col("bureau_recent_3m_delinquency_rate") < 0) |
        (F.col("bureau_recent_3m_delinquency_rate") > 1) |
        (F.col("bureau_recent_6m_delinquency_rate") < 0) |
        (F.col("bureau_recent_6m_delinquency_rate") > 1) |
        (F.col("pos_recent_3m_dpd_rate") < 0) |
        (F.col("pos_recent_3m_dpd_rate") > 1) |
        (F.col("cc_recent_3m_dpd_rate") < 0) |
        (F.col("cc_recent_3m_dpd_rate") > 1) |
        (F.col("installment_recent_3m_late_rate") < 0) |
        (F.col("installment_recent_3m_late_rate") > 1)
    )
    .count()
)

risk_relationship_corrected = (
    temporal_deterioration
    .groupBy("TARGET")
    .agg(
        F.count("*").alias("applications"),
        F.avg("any_recent_delinquency_flag").alias(
            "recent_delinquency_share"
        ),
        F.avg("recent_deterioration_flag").alias(
            "deterioration_share"
        ),
        F.avg("bureau_delinquency_deterioration").alias(
            "avg_bureau_delinquency_deterioration"
        ),
        F.avg("installment_late_payment_deterioration").alias(
            "avg_installment_late_deterioration"
        ),
        F.avg("pos_dpd_deterioration").alias(
            "avg_pos_dpd_deterioration"
        ),
        F.avg("cc_dpd_deterioration").alias(
            "avg_cc_dpd_deterioration"
        ),
        F.avg("cc_utilization_deterioration_clean").alias(
            "avg_cc_utilization_deterioration"
        )
    )
    .orderBy("TARGET")
)

print("=" * 70)
print("CORRECTED TEMPORAL DETERIORATION QUALITY GATE")
print("=" * 70)

print("\nRate range checks:")
rate_range_check.show(truncate=False)

print("\nInvalid rate observations:", f"{invalid_rates:,}")

print("\nRisk relationship using corrected rates:")
risk_relationship_corrected.show(truncate=False)

print("\nCorrected deterioration distributions:")
(
    temporal_deterioration
    .select(
        "bureau_delinquency_deterioration",
        "bureau_severe_delinquency_deterioration",
        "installment_late_payment_deterioration",
        "installment_completion_deterioration",
        "pos_dpd_deterioration",
        "pos_severe_dpd_deterioration",
        "cc_dpd_deterioration",
        "cc_severe_dpd_deterioration",
        "cc_utilization_deterioration_clean"
    )
    .describe()
    .show(truncate=False)
)

CORRECTED TEMPORAL DETERIORATION QUALITY GATE

Rate range checks:
+-------------------------+-------------------------+-------------------------+-------------------------+----------------------+----------------------+---------------------+---------------------+------------------------------+------------------------------+
|min_bureau_recent_3m_rate|max_bureau_recent_3m_rate|min_bureau_recent_6m_rate|max_bureau_recent_6m_rate|min_pos_recent_3m_rate|max_pos_recent_3m_rate|min_cc_recent_3m_rate|max_cc_recent_3m_rate|min_installment_recent_3m_rate|max_installment_recent_3m_rate|
+-------------------------+-------------------------+-------------------------+-------------------------+----------------------+----------------------+---------------------+---------------------+------------------------------+------------------------------+
|0.0                      |1.0                      |0.0                      |1.0                      |0.0                   |1.0                   |0.0      

In [0]:
# Temporal risk profiling across deterioration and recency bands

# -------------------------------------------------------------------
# Create interpretable temporal indicators
# -------------------------------------------------------------------

temporal_analysis = (
    temporal_deterioration
    .withColumn(
        "bureau_recent_delinquency_flag",
        F.when(
            F.coalesce(
                F.col("bureau_recent_3m_delinquent_records"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "bureau_recent_severe_delinquency_flag",
        F.when(
            F.coalesce(
                F.col("bureau_recent_3m_severe_delinquent_records"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "pos_recent_dpd_flag",
        F.when(
            F.coalesce(
                F.col("pos_recent_3m_dpd_records"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "cc_recent_dpd_flag",
        F.when(
            F.coalesce(
                F.col("cc_recent_3m_dpd_records"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "installment_recent_late_flag",
        F.when(
            F.coalesce(
                F.col("installment_recent_3m_late_count"),
                F.lit(0)
            ) > 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "utilization_increase_flag",
        F.when(
            F.col("cc_utilization_deterioration_clean") > 0,
            1
        ).otherwise(0)
    )
)

# -------------------------------------------------------------------
# Overall temporal risk profile
# -------------------------------------------------------------------

overall_temporal_profile = (
    temporal_analysis
    .groupBy("TARGET")
    .agg(
        F.count("*").alias("applications"),

        F.avg(
            "bureau_recent_delinquency_flag"
        ).alias("bureau_recent_delinquency_share"),

        F.avg(
            "bureau_recent_severe_delinquency_flag"
        ).alias("bureau_recent_severe_delinquency_share"),

        F.avg(
            "pos_recent_dpd_flag"
        ).alias("pos_recent_dpd_share"),

        F.avg(
            "cc_recent_dpd_flag"
        ).alias("cc_recent_dpd_share"),

        F.avg(
            "installment_recent_late_flag"
        ).alias("installment_recent_late_share"),

        F.avg(
            "utilization_increase_flag"
        ).alias("cc_utilization_increase_share"),

        F.avg(
            "bureau_delinquency_deterioration"
        ).alias("avg_bureau_deterioration"),

        F.avg(
            "installment_late_payment_deterioration"
        ).alias("avg_installment_late_deterioration"),

        F.avg(
            "pos_dpd_deterioration"
        ).alias("avg_pos_dpd_deterioration"),

        F.avg(
            "cc_dpd_deterioration"
        ).alias("avg_cc_dpd_deterioration"),

        F.avg(
            "cc_utilization_deterioration_clean"
        ).alias("avg_cc_utilization_deterioration")
    )
    .orderBy("TARGET")
)

# -------------------------------------------------------------------
# Risk profile by recent delinquency presence
# -------------------------------------------------------------------

recent_delinquency_risk = (
    temporal_analysis
    .groupBy("any_recent_delinquency_flag")
    .agg(
        F.count("*").alias("applications"),
        F.avg("TARGET").alias("default_rate"),
        F.avg(
            "recent_deterioration_flag"
        ).alias("deterioration_share"),
        F.avg(
            "cc_utilization_deterioration_clean"
        ).alias("avg_cc_utilization_deterioration")
    )
    .orderBy("any_recent_delinquency_flag")
)

# -------------------------------------------------------------------
# Risk profile by deterioration presence
# -------------------------------------------------------------------

deterioration_risk = (
    temporal_analysis
    .groupBy("recent_deterioration_flag")
    .agg(
        F.count("*").alias("applications"),
        F.avg("TARGET").alias("default_rate"),
        F.avg(
            "any_recent_delinquency_flag"
        ).alias("recent_delinquency_share"),
        F.avg(
            "cc_utilization_deterioration_clean"
        ).alias("avg_cc_utilization_deterioration")
    )
    .orderBy("recent_deterioration_flag")
)

# -------------------------------------------------------------------
# Credit-card utilization trajectory bands
# -------------------------------------------------------------------

utilization_band = (
    temporal_analysis
    .withColumn(
        "utilization_deterioration_band",
        F.when(
            F.col("cc_utilization_deterioration_clean").isNull(),
            "No CC history"
        )
        .when(
            F.col("cc_utilization_deterioration_clean") <= -0.10,
            "Large decrease"
        )
        .when(
            F.col("cc_utilization_deterioration_clean") <= -0.02,
            "Moderate decrease"
        )
        .when(
            F.col("cc_utilization_deterioration_clean") < 0.02,
            "Stable"
        )
        .when(
            F.col("cc_utilization_deterioration_clean") < 0.10,
            "Moderate increase"
        )
        .otherwise(
            "Large increase"
        )
    )
    .groupBy("utilization_deterioration_band")
    .agg(
        F.count("*").alias("applications"),
        F.avg("TARGET").alias("default_rate"),
        F.avg("any_recent_delinquency_flag").alias(
            "recent_delinquency_share"
        ),
        F.avg("recent_deterioration_flag").alias(
            "deterioration_share"
        ),
        F.avg("cc_historical_avg_utilization_clean").alias(
            "avg_historical_utilization"
        ),
        F.avg("cc_recent_3m_avg_utilization_clean").alias(
            "avg_recent_3m_utilization"
        )
    )
)

# -------------------------------------------------------------------
# Bureau recent delinquency bands
# -------------------------------------------------------------------

bureau_delinquency_band = (
    temporal_analysis
    .withColumn(
        "bureau_recent_delinquency_band",
        F.when(
            F.col("bureau_recent_3m_delinquency_rate").isNull(),
            "No bureau history"
        )
        .when(
            F.col("bureau_recent_3m_delinquency_rate") == 0,
            "No recent delinquency"
        )
        .when(
            F.col("bureau_recent_3m_delinquency_rate") <= 0.25,
            "Low recent delinquency"
        )
        .when(
            F.col("bureau_recent_3m_delinquency_rate") <= 0.50,
            "Moderate recent delinquency"
        )
        .otherwise(
            "High recent delinquency"
        )
    )
    .groupBy("bureau_recent_delinquency_band")
    .agg(
        F.count("*").alias("applications"),
        F.avg("TARGET").alias("default_rate"),
        F.avg("recent_deterioration_flag").alias(
            "deterioration_share"
        )
    )
)

# -------------------------------------------------------------------
# Installment payment deterioration bands
# -------------------------------------------------------------------

installment_deterioration_band = (
    temporal_analysis
    .withColumn(
        "installment_deterioration_band",
        F.when(
            F.col("installment_late_payment_deterioration").isNull(),
            "No installment history"
        )
        .when(
            F.col("installment_late_payment_deterioration") <= -0.10,
            "Improving strongly"
        )
        .when(
            F.col("installment_late_payment_deterioration") < 0,
            "Improving"
        )
        .when(
            F.col("installment_late_payment_deterioration") == 0,
            "Stable"
        )
        .when(
            F.col("installment_late_payment_deterioration") < 0.10,
            "Deteriorating"
        )
        .otherwise(
            "Deteriorating strongly"
        )
    )
    .groupBy("installment_deterioration_band")
    .agg(
        F.count("*").alias("applications"),
        F.avg("TARGET").alias("default_rate"),
        F.avg("any_recent_delinquency_flag").alias(
            "recent_delinquency_share"
        )
    )
)

# -------------------------------------------------------------------
# Display results
# -------------------------------------------------------------------

print("=" * 70)
print("TEMPORAL RISK PROFILE")
print("=" * 70)

print("\nOverall temporal indicators by TARGET:")
overall_temporal_profile.show(truncate=False)

print("\nDefault rate by recent delinquency presence:")
recent_delinquency_risk.show(truncate=False)

print("\nDefault rate by deterioration presence:")
deterioration_risk.show(truncate=False)

print("\nCredit-card utilization trajectory:")
utilization_band.orderBy(
    F.when(
        F.col("utilization_deterioration_band") == "No CC history", 0
    ).when(
        F.col("utilization_deterioration_band") == "Large decrease", 1
    ).when(
        F.col("utilization_deterioration_band") == "Moderate decrease", 2
    ).when(
        F.col("utilization_deterioration_band") == "Stable", 3
    ).when(
        F.col("utilization_deterioration_band") == "Moderate increase", 4
    ).otherwise(5)
).show(truncate=False)

print("\nBureau recent delinquency profile:")
bureau_delinquency_band.show(truncate=False)

print("\nInstallment payment deterioration profile:")
installment_deterioration_band.show(truncate=False)

TEMPORAL RISK PROFILE

Overall temporal indicators by TARGET:
+------+------------+-------------------------------+--------------------------------------+--------------------+--------------------+-----------------------------+-----------------------------+------------------------+----------------------------------+-------------------------+------------------------+--------------------------------+
|TARGET|applications|bureau_recent_delinquency_share|bureau_recent_severe_delinquency_share|pos_recent_dpd_share|cc_recent_dpd_share |installment_recent_late_share|cc_utilization_increase_share|avg_bureau_deterioration|avg_installment_late_deterioration|avg_pos_dpd_deterioration|avg_cc_dpd_deterioration|avg_cc_utilization_deterioration|
+------+------------+-------------------------------+--------------------------------------+--------------------+--------------------+-----------------------------+-----------------------------+------------------------+----------------------------------+------

In [0]:
# Final temporal risk-intelligence feature layer and quality gate

# -------------------------------------------------------------------
# Construct activity recency measures
# -------------------------------------------------------------------

temporal_final = (
    temporal_analysis
    .withColumn(
        "bureau_activity_recency_months",
        F.when(
            F.col("bureau_most_recent_month").isNotNull(),
            -F.col("bureau_most_recent_month")
        )
    )
    .withColumn(
        "pos_activity_recency_months",
        F.when(
            F.col("pos_most_recent_month").isNotNull(),
            -F.col("pos_most_recent_month")
        )
    )
    .withColumn(
        "cc_activity_recency_months",
        F.when(
            F.col("cc_most_recent_month").isNotNull(),
            -F.col("cc_most_recent_month")
        )
    )
    .withColumn(
        "installment_activity_recency_days",
        F.when(
            F.col("installment_most_recent_day").isNotNull(),
            -F.col("installment_most_recent_day")
        )
    )
)

# -------------------------------------------------------------------
# Curated temporal feature set
# -------------------------------------------------------------------

temporal_feature_columns = [
    "SK_ID_CURR",
    "TARGET",

    # History coverage and account depth
    "temporal_history_available",
    "bureau_account_count",
    "bureau_month_record_count",
    "installment_account_count",
    "installment_payment_records",
    "pos_account_count",
    "pos_month_records",
    "cc_account_count",
    "cc_month_records",

    # Activity recency
    "bureau_activity_recency_months",
    "pos_activity_recency_months",
    "cc_activity_recency_months",
    "installment_activity_recency_days",

    # Historical delinquency
    "bureau_historical_delinquency_rate",
    "bureau_historical_severe_delinquency_rate",
    "pos_historical_dpd_rate",
    "pos_historical_severe_dpd_rate",
    "cc_historical_dpd_rate",
    "cc_historical_severe_dpd_rate",

    # Recent delinquency
    "bureau_recent_3m_delinquency_rate",
    "bureau_recent_6m_delinquency_rate",
    "bureau_recent_3m_severe_delinquency_rate",
    "bureau_recent_6m_severe_delinquency_rate",
    "pos_recent_3m_dpd_rate",
    "pos_recent_6m_dpd_rate",
    "pos_recent_3m_severe_dpd_rate",
    "pos_recent_6m_severe_dpd_rate",
    "cc_recent_3m_dpd_rate",
    "cc_recent_6m_dpd_rate",
    "cc_recent_3m_severe_dpd_rate",
    "cc_recent_6m_severe_dpd_rate",

    # Recent repayment performance
    "installment_historical_late_rate",
    "installment_historical_completion_rate",
    "installment_recent_3m_late_rate",
    "installment_recent_6m_late_rate",
    "installment_recent_3m_completion_rate",
    "installment_recent_6m_completion_rate",

    # Deterioration measures
    "bureau_delinquency_deterioration",
    "bureau_severe_delinquency_deterioration",
    "pos_dpd_deterioration",
    "pos_severe_dpd_deterioration",
    "cc_dpd_deterioration",
    "cc_severe_dpd_deterioration",
    "installment_late_payment_deterioration",
    "installment_completion_deterioration",

    # Credit-card utilization trajectory
    "cc_historical_avg_utilization_clean",
    "cc_recent_3m_avg_utilization_clean",
    "cc_recent_6m_avg_utilization",
    "cc_recent_12m_avg_utilization",
    "cc_utilization_deterioration_clean",

    # Composite temporal indicators
    "bureau_recent_delinquency_flag",
    "bureau_recent_severe_delinquency_flag",
    "pos_recent_dpd_flag",
    "cc_recent_dpd_flag",
    "installment_recent_late_flag",
    "utilization_increase_flag",
    "any_recent_delinquency_flag",
    "recent_deterioration_flag"
]

missing_temporal_columns = [
    column_name
    for column_name in temporal_feature_columns
    if column_name not in temporal_final.columns
]

if missing_temporal_columns:
    raise ValueError(
        f"Missing required temporal features: {missing_temporal_columns}"
    )

temporal_features = temporal_final.select(*temporal_feature_columns)

# -------------------------------------------------------------------
# Basic structural quality checks
# -------------------------------------------------------------------

temporal_row_count = temporal_features.count()

temporal_unique_customers = (
    temporal_features
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

temporal_duplicate_customers = (
    temporal_features
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

target_nulls = (
    temporal_features
    .filter(F.col("TARGET").isNull())
    .count()
)

# -------------------------------------------------------------------
# Rate and flag quality checks
# -------------------------------------------------------------------

rate_columns = [
    "bureau_historical_delinquency_rate",
    "bureau_historical_severe_delinquency_rate",
    "pos_historical_dpd_rate",
    "pos_historical_severe_dpd_rate",
    "cc_historical_dpd_rate",
    "cc_historical_severe_dpd_rate",
    "bureau_recent_3m_delinquency_rate",
    "bureau_recent_6m_delinquency_rate",
    "bureau_recent_3m_severe_delinquency_rate",
    "bureau_recent_6m_severe_delinquency_rate",
    "pos_recent_3m_dpd_rate",
    "pos_recent_6m_dpd_rate",
    "pos_recent_3m_severe_dpd_rate",
    "pos_recent_6m_severe_dpd_rate",
    "cc_recent_3m_dpd_rate",
    "cc_recent_6m_dpd_rate",
    "cc_recent_3m_severe_dpd_rate",
    "cc_recent_6m_severe_dpd_rate",
    "installment_historical_late_rate",
    "installment_historical_completion_rate",
    "installment_recent_3m_late_rate",
    "installment_recent_6m_late_rate",
    "installment_recent_3m_completion_rate",
    "installment_recent_6m_completion_rate",
    "cc_historical_avg_utilization_clean",
    "cc_recent_3m_avg_utilization_clean",
    "cc_recent_6m_avg_utilization",
    "cc_recent_12m_avg_utilization"
]

invalid_rate_condition = None

for column_name in rate_columns:
    condition = (
        (F.col(column_name) < 0) |
        (F.col(column_name) > 1)
    )

    if invalid_rate_condition is None:
        invalid_rate_condition = condition
    else:
        invalid_rate_condition = invalid_rate_condition | condition

invalid_rate_count = (
    temporal_features
    .filter(invalid_rate_condition)
    .count()
)

invalid_flag_count = (
    temporal_features
    .filter(
        (F.col("bureau_recent_delinquency_flag").isin(0, 1) == False) |
        (F.col("bureau_recent_severe_delinquency_flag").isin(0, 1) == False) |
        (F.col("pos_recent_dpd_flag").isin(0, 1) == False) |
        (F.col("cc_recent_dpd_flag").isin(0, 1) == False) |
        (F.col("installment_recent_late_flag").isin(0, 1) == False) |
        (F.col("utilization_increase_flag").isin(0, 1) == False) |
        (F.col("any_recent_delinquency_flag").isin(0, 1) == False) |
        (F.col("recent_deterioration_flag").isin(0, 1) == False)
    )
    .count()
)

# -------------------------------------------------------------------
# Recency quality checks
# -------------------------------------------------------------------

negative_recency_count = (
    temporal_features
    .filter(
        (F.col("bureau_activity_recency_months") < 0) |
        (F.col("pos_activity_recency_months") < 0) |
        (F.col("cc_activity_recency_months") < 0) |
        (F.col("installment_activity_recency_days") < 0)
    )
    .count()
)

# -------------------------------------------------------------------
# Final temporal feature summary
# -------------------------------------------------------------------

temporal_feature_inventory = spark.createDataFrame(
    [
        ("applications", temporal_row_count),
        ("unique_customers", temporal_unique_customers),
        ("duplicate_customers", temporal_duplicate_customers),
        ("target_nulls", target_nulls),
        ("invalid_rate_observations", invalid_rate_count),
        ("invalid_flag_observations", invalid_flag_count),
        ("negative_recency_observations", negative_recency_count),
        ("feature_count", len(temporal_feature_columns))
    ],
    ["quality_check", "value"]
)

print("=" * 70)
print("FINAL TEMPORAL FEATURE QUALITY GATE")
print("=" * 70)

temporal_feature_inventory.show(truncate=False)

# -------------------------------------------------------------------
# Temporal feature coverage
# -------------------------------------------------------------------

print("\nFeature coverage:")

coverage_columns = [
    "bureau_historical_delinquency_rate",
    "bureau_recent_3m_delinquency_rate",
    "pos_historical_dpd_rate",
    "pos_recent_3m_dpd_rate",
    "cc_historical_dpd_rate",
    "cc_recent_3m_dpd_rate",
    "installment_historical_late_rate",
    "installment_recent_3m_late_rate",
    "cc_historical_avg_utilization_clean",
    "cc_recent_3m_avg_utilization_clean",
    "bureau_activity_recency_months",
    "pos_activity_recency_months",
    "cc_activity_recency_months",
    "installment_activity_recency_days"
]

coverage_expression = [
    F.sum(
        F.when(F.col(column_name).isNotNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in coverage_columns
]

temporal_features.agg(*coverage_expression).show(truncate=False)

# -------------------------------------------------------------------
# Default-rate comparison for key temporal indicators
# -------------------------------------------------------------------

print("\nKey temporal indicators by TARGET:")

(
    temporal_features
    .groupBy("TARGET")
    .agg(
        F.count("*").alias("applications"),

        F.avg(
            "any_recent_delinquency_flag"
        ).alias("recent_delinquency_share"),

        F.avg(
            "recent_deterioration_flag"
        ).alias("deterioration_share"),

        F.avg(
            "utilization_increase_flag"
        ).alias("utilization_increase_share"),

        F.avg(
            "bureau_recent_delinquency_flag"
        ).alias("bureau_recent_delinquency_share"),

        F.avg(
            "installment_recent_late_flag"
        ).alias("installment_recent_late_share"),

        F.avg(
            "cc_recent_dpd_flag"
        ).alias("cc_recent_dpd_share")
    )
    .orderBy("TARGET")
    .show(truncate=False)
)

# -------------------------------------------------------------------
# Persist final temporal feature layer
# -------------------------------------------------------------------

(
    temporal_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TEMPORAL_OUTPUT_PATH)
)

# -------------------------------------------------------------------
# Reload and verify persisted output
# -------------------------------------------------------------------

temporal_features_verified = (
    spark.read
    .format("delta")
    .load(TEMPORAL_OUTPUT_PATH)
)

verified_rows = temporal_features_verified.count()
verified_columns = len(temporal_features_verified.columns)
verified_unique_customers = (
    temporal_features_verified
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

print("\nPersisted temporal dataset:")
print("Path:", TEMPORAL_OUTPUT_PATH)
print("Rows:", f"{verified_rows:,}")
print("Columns:", verified_columns)
print("Unique customers:", f"{verified_unique_customers:,}")

if (
    verified_rows != temporal_row_count
    or verified_unique_customers != temporal_unique_customers
    or temporal_duplicate_customers != 0
    or target_nulls != 0
    or invalid_rate_count != 0
    or invalid_flag_count != 0
    or negative_recency_count != 0
):
    raise ValueError(
        "Final temporal feature quality gate failed."
    )

print("\nFINAL TEMPORAL FEATURE QUALITY GATE: PASSED")

FINAL TEMPORAL FEATURE QUALITY GATE
+-----------------------------+------+
|quality_check                |value |
+-----------------------------+------+
|applications                 |307511|
|unique_customers             |307511|
|duplicate_customers          |0     |
|target_nulls                 |0     |
|invalid_rate_observations    |0     |
|invalid_flag_observations    |0     |
|negative_recency_observations|0     |
|feature_count                |60    |
+-----------------------------+------+


Feature coverage:
+----------------------------------+---------------------------------+-----------------------+----------------------+----------------------+---------------------+--------------------------------+-------------------------------+-----------------------------------+----------------------------------+------------------------------+---------------------------+--------------------------+---------------------------------+
|bureau_historical_delinquency_rate|bureau_recent_3m_deli

In [0]:
# Final temporal risk-intelligence summary

# -------------------------------------------------------------------
# Define key temporal risk indicators
# -------------------------------------------------------------------

temporal_risk_indicators = [
    ("Any recent delinquency", "any_recent_delinquency_flag"),
    ("Recent deterioration", "recent_deterioration_flag"),
    ("Credit-card utilization increase", "utilization_increase_flag"),
    ("Recent bureau delinquency", "bureau_recent_delinquency_flag"),
    ("Recent bureau severe delinquency", "bureau_recent_severe_delinquency_flag"),
    ("Recent POS DPD", "pos_recent_dpd_flag"),
    ("Recent credit-card DPD", "cc_recent_dpd_flag"),
    ("Recent installment late payment", "installment_recent_late_flag")
]

# -------------------------------------------------------------------
# Overall portfolio default rate
# -------------------------------------------------------------------

overall_default_rate = (
    temporal_features_verified
    .agg(F.avg("TARGET").alias("default_rate"))
    .first()["default_rate"]
)

# -------------------------------------------------------------------
# Calculate default-rate lift for each temporal indicator
# -------------------------------------------------------------------

indicator_results = []

for indicator_name, column_name in temporal_risk_indicators:

    indicator_profile = (
        temporal_features_verified
        .groupBy(column_name)
        .agg(
            F.count("*").alias("applications"),
            F.avg("TARGET").alias("default_rate")
        )
        .collect()
    )

    profiles = {
        int(row[column_name]): (
            int(row["applications"]),
            float(row["default_rate"])
        )
        for row in indicator_profile
    }

    flagged_count, flagged_rate = profiles.get(
        1,
        (0, None)
    )

    unflagged_count, unflagged_rate = profiles.get(
        0,
        (0, None)
    )

    if (
        flagged_rate is not None
        and unflagged_rate is not None
        and unflagged_rate > 0
    ):
        default_rate_lift = (
            flagged_rate / unflagged_rate
        )
    else:
        default_rate_lift = None

    portfolio_share = (
        flagged_count / temporal_row_count
        if temporal_row_count > 0
        else None
    )

    indicator_results.append(
        (
            indicator_name,
            column_name,
            flagged_count,
            float(portfolio_share) if portfolio_share is not None else None,
            float(flagged_rate) if flagged_rate is not None else None,
            float(unflagged_rate) if unflagged_rate is not None else None,
            float(default_rate_lift)
            if default_rate_lift is not None
            else None
        )
    )

# -------------------------------------------------------------------
# Explicit schema for Spark DataFrame construction
# -------------------------------------------------------------------

temporal_summary_schema = T.StructType([
    T.StructField(
        "indicator",
        T.StringType(),
        False
    ),
    T.StructField(
        "feature",
        T.StringType(),
        False
    ),
    T.StructField(
        "flagged_applications",
        T.LongType(),
        False
    ),
    T.StructField(
        "flagged_portfolio_share",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "flagged_default_rate",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "unflagged_default_rate",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "default_rate_lift",
        T.DoubleType(),
        True
    )
])

temporal_indicator_summary = spark.createDataFrame(
    indicator_results,
    schema=temporal_summary_schema
).orderBy(
    F.desc("default_rate_lift")
)

# -------------------------------------------------------------------
# Overall portfolio reference row
# -------------------------------------------------------------------

overall_reference = [
    (
        "Overall application-level TARGET",
        "TARGET",
        int(temporal_row_count),
        1.0,
        float(overall_default_rate),
        None,
        1.0
    )
]

overall_reference_df = spark.createDataFrame(
    overall_reference,
    schema=temporal_summary_schema
)

# -------------------------------------------------------------------
# Combine overall reference with indicator results
# -------------------------------------------------------------------

final_temporal_risk_summary = (
    overall_reference_df
    .unionByName(temporal_indicator_summary)
)

# -------------------------------------------------------------------
# Display final risk-intelligence summary
# -------------------------------------------------------------------

print("=" * 70)
print("FINAL TEMPORAL RISK-INTELLIGENCE SUMMARY")
print("=" * 70)

final_temporal_risk_summary.show(
    truncate=False
)

# -------------------------------------------------------------------
# Validate summary
# -------------------------------------------------------------------

key_indicator_count = temporal_indicator_summary.count()

positive_lift_count = (
    temporal_indicator_summary
    .filter(
        F.col("default_rate_lift") > 1
    )
    .count()
)

print(
    "\nOverall TARGET default rate:",
    f"{overall_default_rate:.4%}"
)

print(
    "Temporal indicators evaluated:",
    key_indicator_count
)

print(
    "Indicators with >1.0x default-rate lift:",
    positive_lift_count
)

# -------------------------------------------------------------------
# Persist compact interpretation output
# -------------------------------------------------------------------

TEMPORAL_SUMMARY_PATH = (
    f"{HOME_CREDIT_RAW_PATH}/"
    "home_credit_temporal_risk_summary"
)

(
    final_temporal_risk_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TEMPORAL_SUMMARY_PATH)
)

# -------------------------------------------------------------------
# Reload and verify persisted output
# -------------------------------------------------------------------

verified_temporal_summary = (
    spark.read
    .format("delta")
    .load(TEMPORAL_SUMMARY_PATH)
)

verified_summary_rows = verified_temporal_summary.count()

print("\nPersisted temporal risk summary:")
print("Path:", TEMPORAL_SUMMARY_PATH)
print("Rows:", f"{verified_summary_rows:,}")

if verified_summary_rows != key_indicator_count + 1:
    raise ValueError(
        "Temporal risk summary validation failed."
    )

print("\nTEMPORAL RISK-INTELLIGENCE SUMMARY: PASSED")

FINAL TEMPORAL RISK-INTELLIGENCE SUMMARY
+--------------------------------+-------------------------------------+--------------------+-----------------------+--------------------+----------------------+------------------+
|indicator                       |feature                              |flagged_applications|flagged_portfolio_share|flagged_default_rate|unflagged_default_rate|default_rate_lift |
+--------------------------------+-------------------------------------+--------------------+-----------------------+--------------------+----------------------+------------------+
|Recent POS DPD                  |pos_recent_dpd_flag                  |2111                |0.0068647950805011855  |0.16816674561819042 |0.08012442698100851   |2.0988199473557563|
|Recent credit-card DPD          |cc_recent_dpd_flag                   |1172                |0.003811245776573846   |0.1493174061433447  |0.08046641139391328   |1.8556488795353379|
|Credit-card utilization increase|utilization_increase

## Final Temporal and Deterioration Interpretation

The temporal analysis demonstrates that historical credit behaviour contains meaningful deterioration and recency signals within the Home Credit population. Customers exhibiting recent delinquency, recent payment deterioration or increasing credit-card utilization show higher application-level default rates than customers without these signals.

Recent POS delinquency provides the strongest observed temporal separation, with approximately 2.10x the default rate of customers without recent POS delinquency. Recent credit-card delinquency, increasing credit-card utilization, recent installment lateness and broader recent deterioration also show positive default-rate lift.

These temporal features are used as **risk-intelligence and behavioural-deterioration indicators**, not as a newly manufactured monthly default model. The standard application-level `TARGET` remains the outcome used for portfolio-level interpretation.

The final temporal feature layer contains 60 customer-level features and has passed structural, rate, flag and recency quality checks. The temporal analysis remains separate from the Nigerian BNPL modelling population and is not used to replace or directly compete with the BNPL probability-of-default model.

All findings are based on the Home Credit dataset and should be interpreted within its population and data limitations.